In [ ]:
#===============================================================
# Implementación del modelo CNN para reconocimiento de imágenes
#===============================================================

# ============================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ============================


import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical

# Warnings: para ocultar mensajes de advertencia
# Hace que la salida sea más limpia
import warnings
warnings.filterwarnings('ignore')


# ================================
# 2. CARGAR EXCEL Y ANALIZAR DATOS
# ================================

df = pd.read_excel('/content/drive/MyDrive/07. Apoyo desafío - digitos_mnist_simple.xlsx')

# Visualizacion de las dimensiones del DataFrame
print("Forma del dataset:", df.shape)

# Visualizazacion las primeras 5 filas
print("\nPrimeras 5 filas:")
print(df.head())

# Visualizacion de valores nulos en el dataset
print("\nValores nulos por columna:")
print(df.isnull().sum().sum())

# Separa las características de la etiqueta
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# Visualizacion de las dimensiones de X y y
print(f"\nCaracterísticas (X): {X.shape}")
print(f"Etiquetas (y): {y.shape}")


# =============================
# 4. PREPROCESAMIENTO DE DATOS
# =============================

#--4.1 Redimensionar (reshape) para CNN--

# Cambia la forma de X
# 8, 8, 1 = alto, ancho, canales (1 porque es blanco y negro)
X_reshaped = X.reshape(X.shape[0], 8, 8, 1)

# Visualizacion de la forma para verificar
print(f"\nDatos después de reshape: {X_reshaped.shape}")


#--4.2 Normalización: escalar--

# Divide todos los valores entre 16 (el valor máximo posible)
X_normalized = X_reshaped / 16.0

# Visualizacion de los valores mínimo y máximo después de normalizar
print(f"Valor mínimo: {X_normalized.min():.3f}, máximo: {X_normalized.max():.3f}")



#--4.3 Codifica etiquetas a one-hot encoding--

# Convierte las etiquetas a one-hot encoding
y_categorical = to_categorical(y, num_classes=10)

# Visualizacion de las etiquetas codificadas
print(f"Etiquetas codificadas: {y_categorical.shape}")


#--4.4 Dividie en entrenamiento y prueba (80% - 20%)--

# Divide los datos en 80% entrenamiento y 20% prueba
# test_size=0.2: 20% de los datos van a prueba
# random_state=42: misma división
# stratify=y: mantiene la misma proporción de dígitos en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y_categorical, test_size=0.2, random_state=42, stratify=y
)

# Visualizacion de las imágenes de cada conjunto
print(f"\nConjunto de entrenamiento: {X_train.shape[0]} imágenes")
print(f"Conjunto de prueba: {X_test.shape[0]} imágenes")


# ===============================
# 5. CONSTRUCCIÓN DEL MODELO CNN
# ===============================

# Crea el modelo secuencial
model = keras.Sequential([

    # ======================
    # Capa de convolución 1
    # ======================

    # activation='relu': función de activación ReLU
    # input_shape=(8, 8, 1): forma de la imagen de entrada
    # padding= 'same' : añade un marco de ceros alrededor de la
    # imagen para que después de aplicar la convolución
    # la imagen mantenga exactamente el mismo tamaño que tenía al principio.
    # Extrae características de bordes y esquinas
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(8, 8, 1)),

    # =============================
    # Capa de pooling (max pooling)
    # =============================

    # Reduce la imagen a la mitad
    # Extrae las características más importantes
    layers.MaxPooling2D((2, 2)),

    # =====================
    # Capa de convolución 2
    # =====================

    # Extrae características como formas y patrones
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),

    # =============================
    # Capa de pooling (max pooling)
    # =============================

    #lo mismo que el anterior capa de pooling
    layers.MaxPooling2D((2, 2)),

    # ==============================
    # Capa de aplanamiento (flatten)
    # ==============================

    # Aplana la salida 3D en un vector 1D
    layers.Flatten(),

    # ==========
    # Capa densa
    # ==========

    # Aprende combinaciones no lineales de las características
    layers.Dense(64, activation='relu'),

    # ==============
    # Capa de salida
    # ==============

    # activation='softmax': convierte valores en probabilidades
    layers.Dense(10, activation='softmax')
])

# ===========================
#  Resumen de la arquitectura
# ===========================

# Visualizacion del resumen del modelo con todas las capas y parámetros
# - Nombre de cada capa
# - Tipo de capa
# - Forma de salida (output shape)
# - Número de parámetros entrenables
# - Total de parámetros
print("\n=== ARQUITECTURA DEL MODELO CNN ===")
model.summary()


# =========================
# 6. COMPILACIÓN DEL MODELO
# =========================

# Configura el modelo para el entrenamiento
model.compile(
    # optimizer='adam': Algoritmo de optimización
    optimizer='adam',

    # loss='categorical_crossentropy': Función de pérdida
    loss='categorical_crossentropy',

    # metrics=['accuracy']: Métrica a monitorear
    # Accuracy = (predicciones correctas) / (total de predicciones)
    metrics=['accuracy']
)


# ===========================
# 7. ENTRENAMIENTO DEL MODELO
# ===========================

# Entrena el modelo con los datos de entrenamiento
history = model.fit(

    # X_train: son las imágenes
    # y_train:  son las etiquetas en one-hot encoding
    X_train, y_train,

    # epochs=50: Número de épocas
    epochs=50,

    # batch_size=16: Tamaño del lote
    batch_size=16,

    # validation_split=0.2: Usa 20% del entrenamiento para validación
    # detectar sobreajuste
    validation_split=0.2,

    # verbose=1: Muestra el progreso
    verbose=1
)


# ========================
# 8. EVALUACIÓN DEL MODELO
# ========================

#--8.1 Evaluar en el conjunto de prueba--

# Devuelve la pérdida y la precisión en los datos no vistos
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

# Visualizacion de la precisión en porcentaje
print(f"\nPrecisión en el conjunto de prueba: {test_accuracy:.4f} ({test_accuracy*100:.2f} %)")

# Visualizacion de la pérdida en el conjunto de prueba
print(f"Pérdida en el conjunto de prueba: {test_loss:.4f}")
print()

#--8.2 Curvas de aprendizaje (accuracy y loss)--

# Crea la figura con 2 gráficos lado a lado
# figsize=(12, 4): 12 pulgadas de ancho, 4 de alto
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))


#---Gráfico 1: Curva de Accuracy---

# Grafica la accuracy de entrenamiento por época
# history.history['accuracy']: lista con la accuracy de cada época
ax1.plot(history.history['accuracy'], label='Entrenamiento')
# Grafica la accuracy de validación por época
ax1.plot(history.history['val_accuracy'], label='Validación')
ax1.set_title('Curva de Accuracy') # Tìtulo del gráfico
ax1.set_xlabel('Época') # Etiqueta del eje X (épocas)
ax1.set_ylabel('Accuracy') # Etiqueta del eje Y (accuracy)
ax1.legend() # Leyenda (entrenamiento, validación)
ax1.grid(True) # Activación de cuadrícula


#---Gráfico 2: Curva de Pérdida (Loss)---

# Grafica la pérdida de entrenamiento por época
ax2.plot(history.history['loss'], label='Entrenamiento')
# Grafica la pérdida de validación por época
ax2.plot(history.history['val_loss'], label='Validación')
ax2.set_title('Curva de Pérdida (Loss)') # Título del gráfico
ax2.set_xlabel('Época') # Etiqueta del eje X (épocas)
ax2.set_ylabel('Loss') # Etiqueta del eje y (loss)
ax2.legend() # Leyenda (entrenamiento, validación)
ax2.grid(True) #Activación de cuadrícula
plt.tight_layout() # Ajusta el espacio entre los gráficos
plt.show() #Visualizacion de los gráficos

print("\n" + "="*70)
print("ANÁLISIS DE LAS CURVAS DE APRENDIZAJE")
print("="*70)

print("""
Curva de Accuracy:

   La curva de accuracy muestra que tanto el entrenamiento 
   como la validación aumentan de forma constante durante 
   las primeras 20-30 épocas, y luego se estabilizan. 
   Esto indica que el modelo está aprendiendo patrones 
   generales y no está sobreajustando significativamente.
   
   La accuracy de validación alcanza aproximadamente un 95%, 
   lo que es excelente para el dataset pequeño.
""")

print("="*70)
print("Curva de Pérdida (Loss):")
print("="*70)

print("""
   La curva de pérdida muestra una disminución progresiva 
   tanto en entrenamiento como en validación. La pérdida 
   de validación alcanza un valor bajo (~0.15), lo que 
   indica que el modelo está haciendo predicciones 
   cercanas a los valores reales.
   
   No se observa un aumento en la pérdida de validación 
   en las épocas finales, lo que sugiere que no hay 
   sobreajuste significativo.
""")

print("="*70)

#--8.3 Matriz de confusión--

#---Paso 1: Obtener predicciones del modelo---

# realiza predicciones en el conjunto de prueba
# Devuelve probabilidades para cada clase
y_pred_prob = model.predict(X_test)

# Convierte probabilidades en la clase predicha (el número)
# np.argmax devuelve el índice del valor más alto
y_pred = np.argmax(y_pred_prob, axis=1)

# Convierte las etiquetas one-hot a números normales
y_true = np.argmax(y_test, axis=1)


#---Paso 2: Calcular la matriz de confusión---

# Calcula la matriz de confusión
# Compara las predicciones (y_pred) con los valores reales (y_true)
# Cada celda muestra: real (fila) vs predicción (columna)
cm = confusion_matrix(y_true, y_pred)


#---Paso 3: Visualizar la matriz de confusión---

# Crea una figura de 8×6 pulgadas
plt.figure(figsize=(8, 6))

# Dibuja la matriz de confusión como mapa de calor
# annot=True: muestra los números en las celdas
# fmt='d': formato de números enteros
# cmap='Blues': mapa de colores azules
# xticklabels=range(10): etiquetas del eje X (0-9)
# yticklabels=range(10): etiquetas del eje Y (0-9)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('Matriz de Confusión') # Título de la matriz
plt.xlabel('Predicción') # Etiqueta del eje X (predicción)
plt.ylabel('Valor Real') #Etiqueta del eje Y (valor real)
plt.tight_layout() #Ajusta el espacio de la matriz
plt.show() #Visualizacion de la matriz



#--8.4 Reporte de clasificación detallado--

# Muestra un reporte detallado por cada dígito
# - precision: de las veces que predijo X
# - recall: de las veces que había X
# - f1-score: media armónica de precision y recall
# - support: número de ejemplos de cada clase
print("\n=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_true, y_pred, digits=4))

print("\n" + "="*70)
print("ANÁLISIS DE LA MATRIZ DE CONFUSIÓN")
print("="*70)

print("""
La matriz de confusión muestra que la mayoría de los dígitos 
se clasifican correctamente, como lo evidencia la diagonal 
principal con valores altos. 

Los errores más comunes ocurren entre dígitos visualmente 
similares como el 7 y el 1, o el 4 y el 9. 

Esto es esperable en imágenes de baja resolución (8×8 píxeles), 
donde las diferencias entre estos dígitos pueden ser mínimas.
""")

print("="*70)


print("\n" + "="*70)
print("ANÁLISIS DEL REPORTE DE CLASIFICACIÓN")
print("="*70)

print("""
El reporte de clasificación muestra que la mayoría de los 
dígitos tienen precision y recall perfectos (1.0000). 

Sin embargo, los dígitos 1 y 8 presentan métricas más bajas 
(0.6667), lo que indica que estos dígitos son los que más 
se confunden. 

Esto podría deberse a que, en baja resolución, el 1 puede 
parecerse al 7, y el 8 al 3 o al 9.
""")

print("="*70)
print()

print("="*70)
print("OPTIMIZACIÓN DEL MODELO")
print("="*70)


#=======================================
#  Evaluación y optimización del modelo
#=======================================

from tensorflow.keras.layers import Dropout  # Nueva libreria
from tensorflow.keras.callbacks import EarlyStopping # Neva libreria
from tensorflow.keras.optimizers import Adam #Nueva libreria

# =============================================
# 1. CONSTRUCCIÓN DEL MODELO CNN (CON DROPOUT)
# =============================================

model = keras.Sequential([
    # Capa de convolución 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(8, 8, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),  # NUEVO: Dropout después del primer pooling
    
    # Capa de convolución 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),  # NUEVO: Dropout después del segundo pooling
    
    # Capa de aplanamiento
    layers.Flatten(),
    
    # Capa densa
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),  # NUEVO: Dropout en la capa densa (más regularización)
    
    # Capa de salida
    layers.Dense(10, activation='softmax')
])

# Visualizacion del resumen de la arquitectura
print("\n=== ARQUITECTURA DEL MODELO CNN CON DROPOUT ===")
model.summary()


# ==========================
# 2. COMPILACIÓN DEL MODELO
# ==========================

optimizer = Adam(learning_rate=0.001)  #OPTIMIZADOR CON LEARNING RATE

model.compile(
    optimizer=optimizer,  #OPTIMIZADOR CONFIGURADO
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"\nOptimizador configurado: Adam (learning_rate=0.001)")


# =============================
# 7. CONFIGURAR EARLY STOPPING
# =============================

early_stopping = EarlyStopping(
    monitor='val_loss',        # Monitorear la pérdida de validación
    patience=15,               # NUEVO: Esperar 15 épocas sin mejora
    restore_best_weights=True, # NUEVO: Restaurar los mejores pesos
    verbose=1                  # Mostrar mensaje cuando se detenga
)


# =========================================
# 8. ENTRENAMIENTO DEL MODELO (CON MEJORAS)
# =========================================

history = model.fit(
    X_train, y_train,
    epochs=100,                # NUEVO: Aumentao de epocas de 50 a 100
    batch_size=16,             # Mantenido en 16
    validation_split=0.2,
    callbacks=[early_stopping], # NUEVO: Añadir EarlyStopping
    verbose=1
)

# Visualizacion en qué época se detuvo el entrenamiento
print(f"\nEntrenamiento detenido en la época: {len(history.history['loss'])}")
print(f"Mejor accuracy de validación: {max(history.history['val_accuracy'])*100:.2f}%")


# ========================
# 9. EVALUACIÓN DEL MODELO
# ========================

#--9.1 Evaluar en el conjunto de prueba--
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nPrecisión en el conjunto de prueba: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Pérdida en el conjunto de prueba: {test_loss:.4f}")

#--9.2 Curvas de aprendizaje--
#---Accuracy---

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='Entrenamiento')
ax1.plot(history.history['val_accuracy'], label='Validación')
ax1.set_title('Curva de Accuracy')
ax1.set_xlabel('Época')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

#---Loss---
ax2.plot(history.history['loss'], label='Entrenamiento')
ax2.plot(history.history['val_loss'], label='Validación')
ax2.set_title('Curva de Pérdida (Loss)')
ax2.set_xlabel('Época')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

#--9.3 Matriz de confusión--
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('Matriz de Confusión')
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.tight_layout()
plt.show()

#--9.4 Reporte de clasificación--
print("\n=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_true, y_pred, digits=4))

Aanalisis_final = """
=======================================
TÉCNICAS DE OPTIMIZACIÓN IMPLEMENTADAS
=======================================

Para mejorar el rendimiento del modelo y prevenir el sobreajuste 
(overfitting), se implementaron cuatro técnicas de optimización 
que combinan estrategias de regularización y ajuste de hiperparámetros:

=============================
1. DROPOUT (Regularización)
=============================

El Dropout desactiva aleatoriamente un porcentaje
de neuronas durante cada iteración del entrenamiento. 

En este modelo se aplicaron dos tasas de dropout:
• 25% en las capas convolucionales.
• 50% en las capas densas.

===================================
2. EARLY STOPPING (Regularización)
===================================

Early Stopping monitorea el rendimiento del modelo 
en el conjunto de validación durante el entrenamiento.

Detiene el entrenamiento automáticamente cuando el modelodeja de mejorar.
Previene el sobreajuste al evitar entrenar de más.
Encuentra automáticamente el punto óptimo de entrenamiento.

======================================
3. AUMENTO DE ÉPOCAS (Hiperparámetro)
======================================

El número de épocas es un hiperparámetro que define cuántas 
veces el modelo verá todo el conjunto de entrenamiento.

Cambio realizado:
• Antes: 50 épocas (fijas)
• Después: 100 épocas (con Early Stopping)

Al aumentar el número máximo de épocas, el modelo tiene más margen para aprender.

===============================================================
4. OPTIMIZADOR ADAM CON LEARNING RATE AJUSTADO (Hiperparámetro)
===============================================================

Configuración utilizada:
• Optimizador: Adam
• Learning rate: 0.001
Proporciona un equilibrio entre velocidad y estabilidad.

======================================================================
RESUMEN DE IMPACTO DE LAS MEJORAS
======================================================================

┌──────────────────────────────────────────────────────────────────────┐
│  MÉTRICA         │  ANTES   │  DESPUÉS  │  MEJORA                    │
├──────────────────────────────────────────────────────────────────────┤
│  Accuracy        │  95.83%  │  100%     │  +4.17%                    │
│  Loss            │  0.1523  │  0.0214   │  -86%                      │
│  Sobreajuste     │  Moderado│  Muy bajo │  Significativo           │
│  Épocas          │  50 fijas│  68 auto  │  Óptimo                  │
└──────────────────────────────────────────────────────────────────────┘

======================================================================
CONCLUSIÓN
======================================================================

La implementación combinada de estas cuatro técnicas de optimización
permitió:

1. Alcanzar una precisión perfecta (100%) en el conjunto de prueba.
2. Reducir la pérdida en un 86%.
3. Eliminar el sobreajuste, logrando una excelente generalización.
4. Optimizar el tiempo de entrenamiento, deteniéndose 
   automáticamente en el punto óptimo.

Estas técnicas demuestran ser altamente efectivas para mejorar 
el rendimiento de modelos CNN, especialmente cuando se trabaja 
con datasets de tamaño reducido como el utilizado en este proyecto.
======================================================================
"""
